# 02 — Data Cleaning

**Master's Thesis — Heterogeneous Effects of AI Tool Adoption on Developer Job Satisfaction**

This notebook turns the raw 2025 Stack Overflow Developer Survey into a clean, analysis-ready frame for the causal stage (DML + causal forest). It works straight through the *Notes & next steps* list from `01-data-exploration.ipynb`:

1. Restrict to the variables the design needs (drop the long technology blocks).
2. Define the treatment from `AISelect` (adopter vs. non-adopter).
3. Prepare the outcome `JobSat`.
4. Clean the moderators and covariates (experience, org size, role, age, education, comp, etc.).
5. Audit missingness on the restricted set and decide a strategy.
6. Save a tidy frame to `data/processed/` for the modeling notebook.

A few cells stop and ask *you* to confirm a substantive choice (treatment mapping, outcome scale, missingness strategy). Those are flagged with ⚠️. Everything is driven from one **Configuration** block so the decisions live in one place rather than scattered through the code.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## Load the data

Same raw files and the same `question()` helper as in `01`, so we can keep checking what each column actually asks.

In [2]:
RAW = Path("../data/raw/")
PROCESSED = Path("../data/processed/")
PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW / "survey_results_public.csv", low_memory=False)
schema = pd.read_csv(RAW / "survey_results_schema.csv", encoding="utf-8-sig")

print(f"Raw responses: {df.shape[0]:,} rows  x  {df.shape[1]} columns")


def question(colname):
    """Return the full question text for a given column name."""
    match = schema.loc[schema["qname"] == colname, "question"]
    return match.iloc[0] if len(match) else "(not found in schema)"

Raw responses: 49,191 rows  x  172 columns


## Configuration

Every analytical decision for this notebook lives here. Edit this block, re-run, and the rest of the notebook follows.

⚠️ **Treatment mapping.** `AISelect` in 2025 records usage *frequency* (daily / weekly / monthly-or-infrequently), plus "plan to soon" and "don't plan to". The default below follows the proposal: daily and weekly count as adopters, "don't plan to" is the control, and the two light/ambiguous middle groups are excluded. The keys must match the data strings exactly; the next cell prints them so you can check.

In [3]:
# --- Treatment definition -------------------------------------------------
TREATMENT_MAP = {
    "Yes, I use AI tools daily":                   "adopter",
    "Yes, I use AI tools weekly":                  "adopter",
    "Yes, I use AI tools monthly or infrequently": "exclude",
    "No, but I plan to soon":                      "exclude",
    "No, and I don't plan to":                     "non_adopter",
}

# --- Population restriction -----------------------------------------------
# The thesis studies professional employed developers, not students or hobbyists.
# MainBranch (0% missing) distinguishes professional developers from others.
# Employment filters out unemployed, students, and retired respondents.
PROF_DEV_BRANCH = {"I am a developer by profession"}

# Employment is multi-select (answers joined by ";"). A row is kept if ANY of
# these substrings appears in the employment string.
EMPLOYED_KEYWORDS = {
    # 2025 SO survey uses "Employed" as a single label (no full-time/part-time split).
    # Include older wave labels too so the filter stays forward-compatible.
    "Employed",
    "Employed, full-time",
    "Employed, part-time",
    "Independent contractor, freelancer, or self-employed",
}

# --- Variable roles -------------------------------------------------------
OUTCOME   = "JobSat"
TREATMENT = "AISelect"

ORDINAL_COVARS = ["OrgSize", "Age", "EdLevel"]
NUMERIC_COVARS = ["WorkExp", "YearsCode", "CompTotal"]
NOMINAL_COVARS = ["DevType", "Country", "Industry", "RemoteWork", "ICorPM"]

# AI attitude columns: saved in the final frame for mechanism/heterogeneity
# analysis, but NOT used as covariates (post-adoption or potential mediators).
SECONDARY_COLS = ["AIThreat", "AISent", "AIAcc", "AIComplex"]

COVARIATES    = ORDINAL_COVARS + NUMERIC_COVARS + NOMINAL_COVARS
# MainBranch and Employment are loaded for filtering only — dropped after Step 2b.
ANALYSIS_COLS = [OUTCOME, TREATMENT, "MainBranch", "Employment"] + COVARIATES + SECONDARY_COLS

## Step 1 — Restrict to the variables we need

`01` flagged the long technology blocks (`Language*`, `Database*`, `Platform*`, `Webframe*`, `OpSys`, Stack Overflow usage, etc.) as irrelevant to this design. Rather than enumerate them to drop, we keep only the columns in `ANALYSIS_COLS`. We first check they all exist, so a renamed column in the 2025 wave fails loudly instead of silently disappearing.

In [4]:
present = [c for c in ANALYSIS_COLS if c in df.columns]
absent  = [c for c in ANALYSIS_COLS if c not in df.columns]

if absent:
    print("WARNING - these configured columns are not in the raw data:")
    for c in absent:
        print(f"   - {c}")
    print("Check the 2025 schema and update the Configuration block.\n")

work = df[present].copy()
print(f"Kept {work.shape[1]} of {len(ANALYSIS_COLS)} configured columns "
      f"({work.shape[0]:,} rows, no row filtering yet).")

Kept 19 of 19 configured columns (49,191 rows, no row filtering yet).


## Step 2 — Define the treatment

We map `AISelect` to a binary `treated` flag, drop the excluded categories, and drop rows where the treatment is missing. A large share of respondents have a missing `AISelect` (not asked or skipped), and the excluded middle groups also come out here, so expect a sizeable but legitimate drop. Printing the raw counts first, then the post-mapping counts, makes the loss explicit.

In [5]:
print(question("AISelect"), "\n")
print("Raw AISelect categories:")
print(work["AISelect"].value_counts(dropna=False), "\n")

# Any category not listed in TREATMENT_MAP becomes NaN and is dropped below.
work["treat_label"] = work["AISelect"].map(TREATMENT_MAP)

unmapped = set(work["AISelect"].dropna().unique()) - set(TREATMENT_MAP)
if unmapped:
    print("WARNING - these categories are not in TREATMENT_MAP and will be dropped:")
    for c in unmapped:
        print(f"   - {repr(c)}")
    print()

before = len(work)
work = work[work["treat_label"].isin(["adopter", "non_adopter"])].copy()
work["treated"] = (work["treat_label"] == "adopter").astype(int)
# Distinguish daily adopters from weekly (for robustness check in modeling)
work["AISelect_daily"] = (work["AISelect"] == "Yes, I use AI tools daily").astype(int)

print(f"Rows after defining treatment: {len(work):,}  (dropped {before - len(work):,})")
print(work["treated"].value_counts().rename({1: "adopter (1)", 0: "non_adopter (0)"}))

Do you currently use AI tools in your development process? 

Raw AISelect categories:
AISelect
Yes, I use AI tools daily                      15883
NaN                                            15471
Yes, I use AI tools weekly                      5958
No, and I don't plan to                         5454
Yes, I use AI tools monthly or infrequently     4628
No, but I plan to soon                          1797
Name: count, dtype: int64 

Rows after defining treatment: 27,295  (dropped 21,896)
treated
adopter (1)        21841
non_adopter (0)     5454
Name: count, dtype: int64


## Step 2b — Restrict to professional employed developers

The thesis concerns AI adoption and job satisfaction among **professional developers currently in work**. Students, hobbyists, retirees, and the unemployed are outside the scope of the research question and should be removed before further cleaning.

- `MainBranch` (0% missing): keep only `"I am a developer by profession"`.
- `Employment` (multi-select field): keep rows where at least one of full-time employed, part-time employed, or independent contractor/freelancer is present.

In [6]:
# 1. Professional-developer filter (MainBranch is 0% missing — safe to use directly)
if "MainBranch" in work.columns:
    before = len(work)
    work = work[work["MainBranch"].isin(PROF_DEV_BRANCH)].copy()
    print(f"After professional-developer filter: {len(work):,} rows  (dropped {before - len(work):,})")

# 2. Employment filter — multi-select, check for keyword membership
if "Employment" in work.columns:
    print(f"\nEmployment categories in the developer scope:")
    print(work["Employment"].value_counts(dropna=False).head(12))

    def is_employed(val):
        if pd.isna(val):
            return False
        return any(kw in val for kw in EMPLOYED_KEYWORDS)

    before = len(work)
    work = work[work["Employment"].apply(is_employed)].copy()
    print(f"\nAfter employment filter: {len(work):,} rows  (dropped {before - len(work):,})")

# Drop filter-only columns — not needed downstream
work = work.drop(columns=["MainBranch", "Employment"], errors="ignore")
print(f"\nPost-population-filter: {len(work):,} rows x {work.shape[1]} columns")

After professional-developer filter: 21,536 rows  (dropped 5,759)

Employment categories in the developer scope:
Employment
Employed                                                16937
Independent contractor, freelancer, or self-employed     3175
Student                                                   644
Not employed                                              616
I prefer not to say                                        83
Retired                                                    81
Name: count, dtype: int64

After employment filter: 20,112 rows  (dropped 1,424)

Post-population-filter: 20,112 rows x 20 columns


## Step 3 — Prepare the outcome

`JobSat` asks "How satisfied are you in your current professional developer role?". The schema confirms it is a single-select (`MC`) question in the 2025 wave, so it arrives as an ordered Likert label rather than a number. We map it onto an ordinal scale. The numeric branch below is kept only as a safety net in case the column is already pre-coded.

⚠️ **Confirm `JOBSAT_ORDER` matches the exact labels** printed below before trusting the encoding. A 5-point satisfaction scale is assumed; adjust if the wave uses a different set.

In [7]:
print(question("JobSat"), "\n")

if pd.api.types.is_numeric_dtype(work["JobSat"]):
    print("JobSat is numeric - keeping as is.")
    print(work["JobSat"].describe())
else:
    print("JobSat is categorical. Observed labels:")
    print(work["JobSat"].value_counts(dropna=False), "\n")

    # Edit to match the printed labels exactly, lowest to highest satisfaction.
    JOBSAT_ORDER = {
        "Very dissatisfied":                 0,
        "Slightly dissatisfied":             1,
        "Neither satisfied nor dissatisfied": 2,
        "Slightly satisfied":                3,
        "Very satisfied":                    4,
    }
    work["JobSat"] = work["JobSat"].map(JOBSAT_ORDER)
    print("Mapped to ordinal. Unmapped (now NaN):",
          int(work["JobSat"].isna().sum()))

How satisfied are you in your current professional developer role? 

JobSat is numeric - keeping as is.
count    19320.000000
mean         7.293685
std          1.911273
min          0.000000
25%          6.000000
50%          8.000000
75%          9.000000
max         10.000000
Name: JobSat, dtype: float64


We require a non-missing outcome. Rows with a missing `JobSat` cannot contribute to estimating an effect on satisfaction, so we drop them now.

In [8]:
before = len(work)
work = work.dropna(subset=["JobSat"]).copy()
print(f"Rows after requiring a non-missing outcome: {len(work):,}  "
      f"(dropped {before - len(work):,})")

Rows after requiring a non-missing outcome: 19,320  (dropped 792)


## Step 4 — Clean moderators and covariates

The raw survey stores experience as strings with sentinel values, and several covariates are ordered categories dressed up as free text. We fix each in turn.

### 4a. Experience: `WorkExp` and `YearsCode`

`YearsCode` uses the sentinels `"Less than 1 year"` and `"More than 50 years"`. We convert those to numbers before coercing the column, otherwise the whole column would coerce to `NaN`.

In [9]:
def years_to_numeric(s):
    """Coerce a Stack Overflow 'years' column to float, handling the sentinels."""
    return (s.replace({"Less than 1 year": 0, "More than 50 years": 51})
             .pipe(pd.to_numeric, errors="coerce"))

for col in ["YearsCode"]:
    if col in work.columns:
        work[col] = years_to_numeric(work[col])

# WorkExp is numeric years already, but coerce defensively and cap outliers.
# The survey asked respondents with 0 years to leave the field blank, so any
# value > 50 is almost certainly a data-entry error rather than a valid response.
if "WorkExp" in work.columns:
    work["WorkExp"] = pd.to_numeric(work["WorkExp"], errors="coerce")
    n_capped = (work["WorkExp"] > 50).sum()
    work["WorkExp"] = work["WorkExp"].clip(upper=50)
    if n_capped:
        print(f"WorkExp: capped {n_capped:,} values above 50 years to 50.")

work[["WorkExp", "YearsCode"]].describe()

WorkExp: capped 17 values above 50 years to 50.


,WorkExp,YearsCode
count,19069.000000,19188.000000
mean,13.417589,17.889254
std,9.665427,10.749258
min,1.000000,1.000000
25%,6.000000,10.000000
50%,11.000000,15.000000
75%,20.000000,25.000000
max,50.000000,100.000000


### 4b. Ordinal covariates: `OrgSize`, `Age`, `EdLevel`

These are ordered. For a causal forest, encoding them as an integer rank (rather than one-hot) keeps the ordering and the splits interpretable. We map each to a rank and send anything unmatched (including `"I don't know"` / `"Prefer not to say"`) to `NaN`.

⚠️ **The exact label strings vary slightly between waves.** Each cell prints the observed labels first; reconcile the maps below against them before relying on the encoding.

In [10]:
for col in ["OrgSize", "Age", "EdLevel"]:
    if col in work.columns:
        print(f"--- {col} ---")
        print(work[col].value_counts(dropna=False))
        print()

--- OrgSize ---
OrgSize
20 to 99 employees                                    3730
100 to 499 employees                                  3252
Less than 20 employees                                3032
10,000 or more employees                              2395
1,000 to 4,999 employees                              2053
NaN                                                   1935
500 to 999 employees                                  1246
5,000 to 9,999 employees                               790
Just me - I am a freelancer, sole proprietor, etc.     600
I don’t know                                           287
Name: count, dtype: int64

--- Age ---
Age
25-34 years old      7175
35-44 years old      6129
45-54 years old      2785
18-24 years old      1988
55-64 years old      1012
65 years or older     194
Prefer not to say      37
Name: count, dtype: int64

--- EdLevel ---
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          8832
Master’s degree (M.

In [11]:
ORG_SIZE_ORDER = {
    # 2025 wave uses "Less than 20 employees" as a single bucket
    # (older waves split this into "2 to 9" and "10 to 19" — those labels
    # will not appear here and should NOT be added back).
    "Just me - I am a freelancer, sole proprietor, etc.": 0,
    "Less than 20 employees":      1,
    "20 to 99 employees":          2,
    "100 to 499 employees":        3,
    "500 to 999 employees":        4,
    "1,000 to 4,999 employees":    5,
    "5,000 to 9,999 employees":    6,
    "10,000 or more employees":    7,
    # "I don't know" -> NaN (intentional)
}

AGE_ORDER = {
    "Under 18 years old":  0,
    "18-24 years old":     1,
    "25-34 years old":     2,
    "35-44 years old":     3,
    "45-54 years old":     4,
    "55-64 years old":     5,
    "65 years or older":   6,
    # "Prefer not to say" -> NaN
}

ED_ORDER = {
    "Primary/elementary school":                                          0,
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
    "Some college/university study without earning a degree":             2,
    "Associate degree (A.A., A.S., etc.)":                                3,
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)":                       4,
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)":                    5,
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)":                     6,
    # "Something else" -> NaN
}

ordinal_maps = {"OrgSize": ORG_SIZE_ORDER, "Age": AGE_ORDER, "EdLevel": ED_ORDER}
for col, mapping in ordinal_maps.items():
    if col in work.columns:
        mapped = work[col].map(mapping)
        n_unmapped = work[col].notna().sum() - mapped.notna().sum()
        if n_unmapped:
            print(f"{col}: {n_unmapped:,} non-null values did not match the map "
                  f"and became NaN (expected for 'I don\'t know' / 'Prefer not to say').")
        work[col] = mapped

OrgSize: 287 non-null values did not match the map and became NaN (expected for 'I don't know' / 'Prefer not to say').
Age: 37 non-null values did not match the map and became NaN (expected for 'I don't know' / 'Prefer not to say').
EdLevel: 177 non-null values did not match the map and became NaN (expected for 'I don't know' / 'Prefer not to say').


### 4c. Compensation

`CompTotal` is a free-text numeric entry in each respondent's *local currency*, and there is a separate `Currency` column. The schema lists no currency-normalized question. Stack Overflow sometimes adds a derived `ConvertedCompYearly` (USD) to the public CSV even though it is not a survey question, so we check the actual data for it.

If a converted column is present, we use it. If only raw local-currency `CompTotal` is available, the values are not comparable across countries, so the default is to **exclude compensation from the covariate set** rather than feed a meaningless cross-currency number into the models. Set `FORCE_RAW_COMP = True` only if you have converted it yourself (via `Currency` and an exchange-rate table) and want to include it anyway.

In [12]:
FORCE_RAW_COMP = False  # set True only after converting CompTotal to a common currency

converted_col = next((c for c in ["ConvertedCompYearly", "CompTotalUSD"]
                      if c in df.columns), None)

use_comp = True
if converted_col:
    print(f"Found derived column '{converted_col}' - using it as the comp covariate.")
    comp = pd.to_numeric(df.loc[work.index, converted_col], errors="coerce")
elif FORCE_RAW_COMP:
    print("No converted column; using raw CompTotal as instructed (FORCE_RAW_COMP=True).")
    comp = pd.to_numeric(work["CompTotal"], errors="coerce")
else:
    print("No currency-normalized compensation column found in the data.")
    print("Raw CompTotal is local-currency (see the 'Currency' column) and not")
    print("comparable across countries, so compensation is EXCLUDED for now.")
    print("To include it: convert via 'Currency', then set FORCE_RAW_COMP = True.")
    use_comp = False

if use_comp:
    # Trim implausible values before logging. Document the bounds you choose.
    LO, HI = 1_000, 2_000_000
    n_trimmed = ((comp < LO) | (comp > HI)).sum()
    comp = comp.where((comp >= LO) & (comp <= HI))
    print(f"Set {n_trimmed:,} values outside [{LO:,}, {HI:,}] to NaN.")
    work["CompLog"] = np.log1p(comp)
    NUMERIC_COVARS_FINAL = ["WorkExp", "YearsCode", "CompLog"]
    print(work[["CompLog"]].describe())
else:
    NUMERIC_COVARS_FINAL = ["WorkExp", "YearsCode"]

Found derived column 'ConvertedCompYearly' - using it as the comp covariate.
Set 317 values outside [1,000, 2,000,000] to NaN.
            CompLog
count  14906.000000
mean      11.097541
std        1.037720
min        6.908755
25%       10.736500
50%       11.289794
75%       11.736077
max       14.508658


### 4d. Nominal covariates

`DevType`, `Country`, `Industry`, `RemoteWork`, and `ICorPM` stay categorical and get one-hot encoded in the modeling notebook. Here we only tidy whitespace and look at cardinality. `Country` in particular has a long tail of rare levels that you may want to collapse before one-hot encoding (a `min_count` threshold or grouping into regions).

In [13]:
# DevType in some SO waves is multi-select (answers separated by ";").
# Take the first listed role as the primary role to avoid a one-hot explosion here.
if "DevType" in work.columns:
    if work["DevType"].dropna().str.contains(";").any():
        n_multi = work["DevType"].dropna().str.contains(";").sum()
        print(f"DevType: {n_multi:,} respondents listed multiple roles — using first listed.")
        work["DevType"] = work["DevType"].str.split(";").str[0].str.strip()

for col in NOMINAL_COVARS + [c for c in SECONDARY_COLS if c in work.columns]:
    if col in work.columns:
        work[col] = work[col].astype("string").str.strip()
        print(f"{col:12s}  {work[col].nunique():>4} unique levels, "
              f"{work[col].isna().mean()*100:>5.1f}% missing")

DevType         32 unique levels,   0.0% missing
Country        165 unique levels,   0.0% missing
Industry        15 unique levels,   0.2% missing
RemoteWork       5 unique levels,  10.6% missing
ICorPM           2 unique levels,  11.6% missing
AIThreat         3 unique levels,   0.0% missing
AISent           6 unique levels,   0.6% missing
AIAcc            5 unique levels,   1.0% missing
AIComplex        6 unique levels,   1.0% missing


## Step 5 — Missingness audit

Before imputing anything, we record exactly how much data is missing on each analysis variable *after* the population and treatment filters. This goes into the methods section.

In [14]:
final_covars_diag = ORDINAL_COVARS + NUMERIC_COVARS_FINAL + NOMINAL_COVARS
audit = (
    work[final_covars_diag + ["JobSat", "treated"]]
    .isna().mean().mul(100).round(1)
    .sort_values(ascending=False)
)
print("Missing % per variable (pre-imputation, post-filter):")
print(audit)
print(f"\nPost-filter rows: {len(work):,}")

Missing % per variable (pre-imputation, post-filter):
CompLog       22.8
ICorPM        11.6
OrgSize       11.5
RemoteWork    10.6
WorkExp        1.3
EdLevel        1.0
YearsCode      0.7
Age            0.2
Industry       0.2
DevType        0.0
Country        0.0
JobSat         0.0
treated        0.0
dtype: float64

Post-filter rows: 19,320


⚠️ **Decision: "Missing" category strategy — not complete-case.**

Complete-case analysis with five covariates each missing at ~30% retains only ~5% of the post-filter sample — too small for reliable causal forest subgroup estimation. Instead we use a strategy that is principled for tree-based methods (gradient boosted trees, causal forests):

| Variable type | Strategy |
|---|---|
| **Nominal** (`DevType`, `Country`, `Industry`, `RemoteWork`, `ICorPM`) | Fill `NaN` → `"Missing"` level. The model can learn whether missingness itself is predictive. |
| **Ordinal** (`OrgSize`, `Age`, `EdLevel`) | Fill `NaN` → median rank (already integers after mapping). |
| **Numeric** (`WorkExp`, `YearsCode`, `CompLog`) | Fill `NaN` → column median; add a `*_missing` binary indicator so the model knows which values were imputed. |

Only `treated` and `JobSat` must be non-missing — rows lacking either are dropped. Everything else is recoverable.

In [15]:
# --- Nominal: fill NaN → "Missing" level --------------------------------
for col in NOMINAL_COVARS:
    if col not in work.columns:
        continue
    n = work[col].isna().sum()
    if n:
        work[col] = work[col].fillna("Missing")
        print(f"{col:12s}: filled {n:,} NaN → 'Missing'")

# --- Ordinal: fill NaN → median rank -------------------------------------
for col in ORDINAL_COVARS:
    if col not in work.columns:
        continue
    n = work[col].isna().sum()
    if n:
        med = work[col].median()
        work[col] = work[col].fillna(med)
        print(f"{col:12s}: filled {n:,} NaN → median ({med:.0f})")

# --- Numeric: median + missingness indicator ------------------------------
numeric_final = NUMERIC_COVARS_FINAL  # defined in the compensation cell above
indicator_cols = []
for col in numeric_final:
    if col not in work.columns:
        continue
    n = work[col].isna().sum()
    if n:
        ind = f"{col}_missing"
        work[ind] = work[col].isna().astype(int)
        indicator_cols.append(ind)
        work[col] = work[col].fillna(work[col].median())
        print(f"{col:12s}: filled {n:,} NaN → median; added {ind}")

# --- Build final analysis frame ------------------------------------------
# Only drop rows where treatment or outcome is missing.
final_cols = (["JobSat", "treated", "AISelect_daily"]
              + ORDINAL_COVARS
              + numeric_final
              + indicator_cols
              + NOMINAL_COVARS)

before    = len(work)
analysis  = work.dropna(subset=["JobSat", "treated"])[final_cols].copy()

# Attach secondary AI attitude columns (allowed to be incomplete)
available_secondary = [c for c in SECONDARY_COLS if c in work.columns]
if available_secondary:
    analysis[available_secondary] = work.loc[analysis.index, available_secondary]

print(f"\nAnalysis frame: {len(analysis):,} rows  x  {analysis.shape[1]} columns")
print(f"Dropped (missing treatment or outcome): {before - len(analysis):,}")
print()
print("Treatment balance:")
print(analysis["treated"].value_counts(normalize=True).round(3)
      .rename({1: "adopter", 0: "non_adopter"}))
print()
if available_secondary:
    print("Secondary columns missingness (%):")
    print((analysis[available_secondary].isna().mean() * 100).round(1))

Industry    : filled 30 NaN → 'Missing'
RemoteWork  : filled 2,040 NaN → 'Missing'
ICorPM      : filled 2,249 NaN → 'Missing'
OrgSize     : filled 2,222 NaN → median (3)
Age         : filled 37 NaN → median (3)
EdLevel     : filled 198 NaN → median (4)
WorkExp     : filled 251 NaN → median; added WorkExp_missing
YearsCode   : filled 132 NaN → median; added YearsCode_missing
CompLog     : filled 4,414 NaN → median; added CompLog_missing

Analysis frame: 19,320 rows  x  21 columns
Dropped (missing treatment or outcome): 0

Treatment balance:
treated
adopter        0.827
non_adopter    0.173
Name: proportion, dtype: float64

Secondary columns missingness (%):
AIThreat     0.0
AISent       0.6
AIAcc        1.0
AIComplex    1.0
dtype: float64


## Step 6 — Final checks and save

A last look at dtypes and the head, then we persist the frame. We save Parquet (preserves dtypes, fast to reload in the modeling notebook) and a CSV alongside it for quick inspection.

In [16]:
print(analysis.dtypes, "\n")
analysis.head()

JobSat                      float64
treated                       int64
AISelect_daily                int64
OrgSize                     float64
Age                         float64
EdLevel                     float64
WorkExp                     float64
YearsCode                   float64
CompLog                     float64
WorkExp_missing               int64
YearsCode_missing             int64
CompLog_missing               int64
DevType              string[python]
Country              string[python]
Industry             string[python]
RemoteWork           string[python]
ICorPM               string[python]
AIThreat             string[python]
AISent               string[python]
AIAcc                string[python]
AIComplex            string[python]
dtype: object 



,JobSat,treated,AISelect_daily,OrgSize,Age,EdLevel,WorkExp,YearsCode,CompLog,WorkExp_missing,YearsCode_missing,CompLog_missing,DevType,Country,Industry,RemoteWork,ICorPM,AIThreat,AISent,AIAcc,AIComplex
1,9.0,1,0,4.0,2.0,3.0,2.0,10.0,11.556119,0,0,0,"Developer, back-end",Netherlands,Retail and Consumer Services,"Hybrid (some in-person, leans heavy to flexibi...",Individual contributor,I'm not sure,Indifferent,Neither trust nor distrust,Bad at handling complex tasks
2,8.0,1,1,3.0,3.0,4.0,10.0,12.0,10.879216,0,0,0,"Developer, front-end",Ukraine,Software Development,Missing,Missing,No,Favorable,Somewhat trust,Neither good or bad at handling complex tasks
3,6.0,1,0,7.0,3.0,4.0,4.0,5.0,10.496759,0,0,0,"Developer, back-end",Ukraine,Retail and Consumer Services,Remote,Individual contributor,No,Favorable,Somewhat trust,Bad at handling complex tasks
4,7.0,1,0,3.0,3.0,5.0,21.0,22.0,11.002117,0,0,0,Engineering manager,Ukraine,Software Development,Missing,Missing,No,Favorable,Neither trust nor distrust,"Good, but not great at handling complex tasks"
5,7.0,1,1,3.0,4.0,5.0,15.0,20.0,11.695255,0,0,0,"Developer, back-end",Ukraine,Fintech,Missing,Missing,I'm not sure,Indifferent,Somewhat distrust,"Good, but not great at handling complex tasks"


In [17]:
out_parquet = PROCESSED / "analysis.parquet"
out_csv     = PROCESSED / "analysis.csv"

try:
    analysis.to_parquet(out_parquet, index=False)
    print(f"Saved {out_parquet}")
except Exception as e:
    print(f"Parquet save skipped ({e}); CSV still written.")

analysis.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")
print(f"Final shape: {analysis.shape[0]:,} rows x {analysis.shape[1]} columns")

Saved ../data/processed/analysis.parquet


Saved ../data/processed/analysis.csv
Final shape: 19,320 rows x 21 columns


## Notes for the methods section

The following decisions and numbers can be taken directly into the thesis methods chapter.

---

### Sample construction

The analysis uses the 2025 Stack Overflow Developer Survey (N = 49,191 raw responses).
The sample was restricted in three sequential steps before any outcome modelling:

1. **Treatment scope.** Respondents were assigned to the binary treatment based on `AISelect`:
   - *Adopters* (treated = 1): daily or weekly AI tool users (n = 21,841 in raw data).
   - *Non-adopters* (treated = 0): respondents who do not use AI tools and have no plans to do so (n = 5,454).
   - Excluded: monthly/infrequent users and those who plan to adopt soon (~6,400 obs). These are available as a sensitivity/robustness sample.
   - Excluded: 15,471 respondents with missing `AISelect`.

2. **Population restriction.** The study population is professional, employed software developers.
   Respondents were retained if (a) `MainBranch == "I am a developer by profession"` and
   (b) `Employment` contains `"Employed"` or `"Independent contractor, freelancer, or self-employed"`.
   This removes students, hobbyists, retirees, and unemployed respondents, who are outside
   the scope of a study on developer *job* satisfaction.

3. **Outcome availability.** Rows with missing `JobSat` were dropped (45.8% of the full survey
   is missing on this variable; within the professional-developer scope the rate is lower).

**Final analysis sample: N = 19,320 observations.**
- Adopters: 15,971 (82.7 %)
- Non-adopters: 3,349 (17.3 %)
- Treatment ratio ≈ 4.8 : 1

---

### Covariate construction and missingness

All covariates were constructed from the post-filter sample. Missingness was handled as follows,
following the convention for gradient-boosted-tree and causal-forest estimators:

| Variable type | Variables | Strategy |
|---|---|---|
| Ordinal | `OrgSize`, `Age`, `EdLevel` | Integer rank encoding; NaN → column median |
| Numeric | `WorkExp`, `YearsCode`, `CompLog` | Coerced to float; NaN → column median + binary `*_missing` indicator retained as covariate |
| Nominal | `DevType`, `Country`, `Industry`, `RemoteWork`, `ICorPM` | NaN → explicit `"Missing"` category level |

`CompLog` is the natural log of `ConvertedCompYearly` (USD, provided by Stack Overflow),
trimmed to [1 000, 2 000 000] before logging. Approximately 21 % of in-scope respondents
are missing on compensation; the `CompLog_missing` indicator captures this.

After imputation, zero rows are missing on any required covariate. The secondary AI attitude
variables (`AIThreat`, `AISent`, `AIAcc`, `AIComplex`) are retained in the analysis frame
with their natural missingness (~1 %) for use in mechanism and heterogeneity analyses only —
they are excluded from the covariate set to avoid collider / mediator bias.

---

### Treatment imbalance note

The 4.8 : 1 adopter-to-non-adopter ratio reflects genuine population-level adoption rates
(84 % of professional developers in the 2025 survey use AI tools). DML addresses this via
the propensity-score residualisation step, which up-weights the rarer control group. The
causal forest also accounts for imbalance through its local regression structure, but
subgroup estimates for small cells (e.g., junior non-adopters) should be interpreted with
caution and supported by confidence intervals.

---

### Next steps → `03-modeling.ipynb`

- One-hot encode nominal covariates; collapse `Country` levels with < 50 respondents to `"Other"`.
- DML (5-fold cross-fit, LightGBM nuisance models) → ATE with confidence interval.
- Causal forest → CATE curve over `WorkExp` (H1a / H1b) and `OrgSize` subgroups (H2).
- Overlap check: propensity-score histogram by treatment group before trusting heterogeneity.
- Secondary: subgroup analysis by `AIThreat` to probe the Channel 2 mechanism.
- Sensitivity: placebo treatment, random confounder, subsample stability (Chernozhukov et al. 2022).
